# Head crops: letting the model look closer

Snake experts tell similar species apart by looking closely at the head: the scales on top of it, the eye, the heat pit of a pit viper. Our model sees the whole snake crop at 320 × 320, where the head can be a few dozen pixels wide. This notebook tries two ways of giving it a closer look, one per run, chosen by `APPROACH`:

- **Approach 1, `"detector"`:** a detector finds the head, and the model reads the head crop as a second view of the snake. It was dropped after notebook 22's pilot, before any head detector was trained (section 2).
- **Approach 2, `"detail"`:** no head labels at all. The model finds the spot it relied on most, cuts it out, looks again, and averages its two looks. This is the idea of WS-DAN (Hu et al. 2019), with the class activation map standing in for WS-DAN's attention maps.

Each approach trains on the experiment subset with the recipe of the background winner, arm A1 of notebook 18 (the detector's box crop, with the snake pasted onto another species' scenery half the time in training), and is scored exactly as the arms were: macro-F1, WHO recall after the safety layer's thresholds are refitted, and top-1 on the same photos as they are and with their scenery swapped. It is then compared with A1's own row, and timed per frame.

A Kaggle session can wait hours in the queue, so one session does everything: a **smoke pass** runs every step on 2,000 photos per set for one epoch, and the **full pass** (20 epochs on the whole subset) starts only if the smoke pass finished. The runner executes every cell even after one fails, so each stage records that it finished and the next one checks.

It reads the cleaned photos, the SAM snake masks, the experiment subset and the front end's predictions (a box and a predicted mask for every photo, from the cropping notebook's Kaggle run). Run it on Kaggle with a T4 GPU and internet on: about 2.5 hours.

## Setup

We pin the libraries whose versions the recorded runs used. The parameters below are the only lines that change between runs.

In [1]:
%pip install -q timm==1.0.31 albumentations==2.0.8 scikit-learn==1.9.1
import hashlib, json, os, platform, statistics, sys, time, warnings
from pathlib import Path
from types import SimpleNamespace

os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"      # no "new version available" check
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # no download bar for the ImageNet weights
warnings.filterwarnings("ignore")

import kagglehub, torch, yaml
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO / "src"))  # the project's code
from serpentguard import data, models, robustness, video
from serpentguard.decide import decision_report
from serpentguard.metrics import write_json
from serpentguard.train import load_checkpoint, subset_hash, train

pd.set_option("display.precision", 3)
PASSED, SMOKE_PASSED, FULL_PASSED = set(), False, False  # what has finished, for the later cells
print(f"torch {torch.__version__}, {torch.cuda.device_count()} GPU(s)")

Note: you may need to restart the kernel to use updated packages.
torch 2.11.0+cu130, 2 GPU(s)


In [2]:
APPROACH = "detail"  # "detector" (approach 1) or "detail" (approach 2)
WINNER = "M4_A1"  # the background winner's run folder in results/m4/arms
RUN_ID = "M4_H2"
SMOKE_PHOTOS = 2000  # photos per set in the smoke pass; the verdicts need WHO-listed photos in it
FULL = True  # False stops after the smoke pass
FRONT_END = 'vikas25s/serpentguard-m4-frontend-masks'

## 1. The starting settings

The run starts from the settings A1 was trained with, read from its saved `config.yaml`: milestone 3's final recipe (MobileViT-S, ArcFace head, the tuned learning rate, margin and augmentation, 20 epochs, seed 1) on the detector's box crop, with the scenery swapped half the time. Its row of the arm table is the score to beat.

In [3]:
ARMS = REPO / "results" / "m4" / "arms"
winner_cfg = yaml.safe_load((ARMS / WINNER / "config.yaml").read_text())
winner_row = json.loads((ARMS / WINNER / "arm_row.json").read_text())
base = data.run_config({key: value for key, value in winner_cfg.items()
                        if key not in ("subset_path", "scenery_bank")})  # both are set per session
assert not base["detail_crop"] and base["views"] == "body", "the winner already looks closer"
pd.Series({name: base[name] for name in ("run_id", "front_end", "mask_source", "box_source",
                                         "scenery_swap", "epochs", "seed", "lr")},
          dtype=object).rename(f"{WINNER}, as trained")

run_id              M4_A1
front_end              F1
mask_source           sam
box_source      predicted
scenery_swap          0.5
epochs                 20
seed                    1
lr                  0.001
Name: M4_A1, as trained, dtype: object

## 2. Approach 1: a head detector

This approach was dropped before a head detector was trained. The idea was a detector that learns a `head` class next to `snake`, with the model reading the body crop and the head crop through the same encoder and averaging their embeddings. It needs a detector that finds the head, so notebook 22 first asked three open-vocabulary detectors (Grounding DINO tiny and base, and OWLv2) for head and tail boxes on 50 training photos. The best drew 4 right head boxes of 50 and 3 right tail boxes, against a pass mark of 40, so no part passed (each box was judged on a contact sheet; the verdicts are in `results/m4/heads/pilot.csv` and `pilot_retry.csv`). The code cell below is left as the notebook was built: it stops if `APPROACH` is `"detector"`.

In [4]:
if APPROACH == "detector":
    raise NotImplementedError("approach 1 needs the head detector, which this notebook does not train yet")

## 3. Approach 2: the detail crop the model picks itself

The encoder turns the 320 × 320 crop into a 10 × 10 grid of 640 features, and the model averages the grid before its embed layer. The embed layer (a linear layer and a batch norm, which is a fixed scale and shift once trained) and the cosine with a species' direction are both linear, so we can apply them at each cell instead of after averaging. Each cell then gets a score for that species: the **class activation map** (Zhou et al. 2016). The average of its 100 cells is the photo's embedding before its scaling to length 1, dotted with the species' direction: the species head's cosine times the length of that embedding. The length is the same for every species, so the averages rank species exactly as the head does, though they are not the cosine itself. The high cells show where the model found the species.

From the map of the model's top species we build the **detail box**:

1. scale the map to run from 0 to 1, and keep the cells at 0.5 or above (WS-DAN uses 0.4 to 0.6);
2. take the box around those cells;
3. widen each side to at least a quarter of the photo, so a single hot cell never gives a sliver, and keep the box inside the photo. A flat map keeps the whole photo.

The box is cut out of the 320 × 320 input, resized back to 320 × 320 and passed through the same encoder. The two embeddings (before their scaling to length 1) are averaged, and the heads read the average.

In training, half of the batches are replaced by their detail crops, so the encoder learns zoomed views too. Those boxes come from the map of the photo's true species, computed without gradients and with the batch norms in their evaluation form, so the extra pass does not shift their running averages.

A small example on made-up 10 × 10 maps:

In [5]:
example = torch.zeros(3, 10, 10)
example[0, 2:4, 6:9] = 1.0  # a hot patch near the top right
example[0, 7, 1] = 0.3      # weaker evidence elsewhere stays out
example[1, 9, 0] = 1.0      # one hot cell in the bottom left corner
example[2] = 1.0            # a flat map
boxes = models.detail_box(example)
pd.DataFrame(boxes.numpy(), columns=["x1", "y1", "x2", "y2"],
             index=["hot patch", "one corner cell", "flat map"]).round(3)

,x1,y1,x2,y2
hot patch,0.6,0.175,0.90,0.425
one corner cell,0.0,0.750,0.25,1.000
flat map,0.0,0.000,1.00,1.000


The run's settings are the winner's, plus the approach's own:

In [6]:
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else REPO / "runs" / "m4_heads"
BANK = WORK / "scenery_bank"
APPROACHES = {"detail": dict(detail_crop=True, detail_threshold=0.5, detail_train_p=0.5)}


def recipe(run_id, epochs=None):
    cfg = {**base, **APPROACHES[APPROACH], "run_id": run_id,
           "scenery_bank": str(BANK) if base["scenery_swap"] > 0 else None}
    return data.run_config(cfg if epochs is None else {**cfg, "epochs": epochs})


full_cfg = recipe(RUN_ID)
pd.Series({name: full_cfg[name] for name in full_cfg if full_cfg[name] != base[name]},
          dtype=object).rename(f"{RUN_ID}, against {WINNER}")

run_id                                 M4_H2
scenery_bank    /kaggle/working/scenery_bank
detail_crop                             True
Name: M4_H2, against M4_A1, dtype: object

## 4. The inputs

Found by handle with `kagglehub`, and checked as in notebook 18: the experiment subset by its fingerprint, and the front end's file byte for byte against milestone 3's, with every mask it lists present. A teammate's account can't read our cropping run, so there `FRONT_END` names that account's private, byte-identical copy of its output (the boxes and the predicted masks). A1's recipe needs both: the boxes crop every photo, and the predicted masks build the scenery bank and cut the snake out in the scenery test, since the validation photos have no SAM masks. The SAM masks, shared with the team, cut the training snakes for the swap.

In [7]:
OWNER, KERNEL = "puneetbajajiitm", "kernel:"
SUBSET_HASH = "40196d19a68e"  # the first 12 digits of the committed subset's SHA-256
FRONT_END_SHA256 = "9dcc0576eb1c60d3e767ba9ca3c34a6a1b92b6b01946e377042dee1b1287b9e3"


def dataset(slug):
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


def download(handle):
    if handle.startswith(KERNEL):
        return Path(kagglehub.notebook_output_download(handle[len(KERNEL):]))
    return Path(kagglehub.dataset_download(handle))


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


DATA = dataset("serpentguard-indian-snakes-clean")
MASKS = dataset("serpentguard-sam-masks")
SUBSET = dataset("serpentguard-m3-subset") / "subset.csv"
PREDS = next(download(FRONT_END).rglob("frontend_preds.csv")).parent

assert subset_hash(pd.read_csv(SUBSET)["path"]).startswith(SUBSET_HASH), "not the experiment subset"
assert sha256(PREDS / "frontend_preds.csv") == FRONT_END_SHA256, "not milestone 3's front end"
preds = pd.read_csv(PREDS / "frontend_preds.csv")
listed = preds["mask_file"].fillna("").loc[lambda masks: masks.ne("")]
assert all((PREDS / name).is_file() for name in listed), "a listed mask is missing"
box_conf = dict(zip(preds["path"], preds["box_conf"]))
print(f"front end: {preds['box_conf'].notna().sum():,} boxes and {len(listed):,} masks "
      f"for {len(preds):,} photos")
PASSED.add("inputs")

front end: 39,514 boxes and 36,486 masks for 39,526 photos


## 5. The scenery bank

The swap pastes the snake onto the scenery of a training photo of another species, taken from a bank of 3,000 training photos whose snake was painted out, built exactly as in notebook 18 (from the predicted masks, the species taking turns with a fixed seed). Built with the same code from the same photos, its `bank.csv` must match A1's fingerprint, so both runs swap from the same sceneries. The photos may not be redistributed, so this notebook shows counts only.

In [8]:
bank_frames = data.make_frames({**base, "run_id": "scenery_bank", "mask_source": "predicted"},
                               DATA, MASKS, SUBSET, PREDS)
bank = data.build_scenery_bank(bank_frames["train"], BANK)
bank_sha256 = sha256(BANK / "bank.csv")
assert bank_sha256 == winner_row["bank_sha256"], "not the bank A1 trained with"
per_class = bank["class_id"].value_counts()
print(f"{len(bank):,} sceneries, {len(per_class)} species, {per_class.min()} to {per_class.max()} "
      f"each; bank.csv sha256 {bank_sha256[:12]}, the same as {WINNER}'s")
PASSED.add("bank")

3,000 sceneries, 184 species, 10 to 17 each; bank.csv sha256 2eed027132bf, the same as M4_A1's


## 6. The steps of one pass

Each step is a function, so the smoke pass and the full pass run exactly the same code:

1. **training**: `train()` trains the run and writes its folder, as for every run since milestone 3. With the detail crop on, every evaluation already uses the two looks, and each evaluation line records the share of training batches that were detail crops;
2. **verdicts**: the safety layer's thresholds refitted on the run's calibration photos with the lower-bound fit, given the dark and blurry flags and the detector's box confidence, exactly as notebook 18 does for the arms. **WHO recall** is the share of WHO-listed validation photos called dangerous;
3. **the scenery test**: notebook 18's test on the validation photos with a usable predicted mask, as they are, with the snake blanked, and with the snake pasted onto another species' scenery;
4. **where the model looks**: the detail box of every validation photo with the best epoch's weights, summarised by how much it zooms. For the check by eye, 20 validation photos, the species taking turns, are drawn with their boxes into `detail_boxes.jpg` next to the run; they are not shown here, since the photos' licences don't cover this notebook;
5. **the row**: the same fields as an arm's row in notebook 18.

In [9]:
MEAN, STD = np.array(data.IMAGENET_MEAN), np.array(data.IMAGENET_STD)
SHEET_PHOTOS, SHEET_COLUMNS = 20, 5
EDGE = 1e-6  # a box side this close to the photo's edge touches it


def verdicts(run_dir):
    quality = robustness.quality_table(run_dir, DATA)
    decision = decision_report(run_dir, DATA, quality=quality, box_conf=box_conf,
                               recall_bound="lower")
    return quality, decision


def scenery_test(run_dir, quality, limit):
    (run_dir / "r1").mkdir(exist_ok=True)
    return write_json(run_dir / "robustness_metrics.json",
                      robustness.robustness_metrics(run_dir, DATA, PREDS, subset_csv=SUBSET,
                                                    quality=quality, out_dir=run_dir / "r1",
                                                    limit=limit))


@torch.inference_mode()
def detail_boxes(model, cfg, frame, keep_images=False):
    """(boxes, top species, and with keep_images the photos as the model saw them) per photo."""
    device = next(model.parameters()).device
    found, top, images = [], [], []
    for batch in data.eval_loader(cfg, frame, num_workers=4):
        pixels = batch["image"].to(device)
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            maps, mask_logits, features = model.encode(pixels)
            best = model.heads(features, mask_logits)["species_logits"].argmax(1)
            found.append(models.detail_box(models.class_activation(model, maps[-1], best),
                                           cfg["detail_threshold"]).cpu())
        top.append(best.cpu())
        if keep_images:
            images.append(batch["image"])
    return (torch.cat(found).numpy(), torch.cat(top).numpy(),
            torch.cat(images) if keep_images else None)


def box_report(run_dir, cfg, limit):
    _, model, _ = load_checkpoint(run_dir, "cuda" if torch.cuda.is_available() else "cpu")
    val = data.make_frames(cfg, DATA, MASKS, SUBSET, PREDS, limit)["val"]
    boxes, _, _ = detail_boxes(model, cfg, val)
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    corner = (((boxes[:, 0] <= EDGE) | (boxes[:, 2] >= 1 - EDGE))
              & ((boxes[:, 1] <= EDGE) | (boxes[:, 3] >= 1 - EDGE)))
    summary = {"photos": len(boxes), "median_area": float(np.median(area)),
               "whole_photo_share": float((area >= 0.9).mean()),
               "corner_share": float(corner.mean()), "smallest_area": float(area.min())}
    sample = video.round_robin(val, SHEET_PHOTOS, seed=42)
    sample_boxes, sample_top, sample_images = detail_boxes(model, cfg, sample, keep_images=True)
    tile_side = cfg["input_size"]
    sheet = Image.new("RGB", (SHEET_COLUMNS * tile_side,
                              -(-len(sample) // SHEET_COLUMNS) * tile_side))
    for i, (image, box) in enumerate(zip(sample_images, sample_boxes)):
        pixels = (image.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)
        tile = Image.fromarray((pixels * 255).astype(np.uint8))
        ImageDraw.Draw(tile).rectangle(list(box * tile_side), outline=(255, 40, 40), width=3)
        sheet.paste(tile, (i % SHEET_COLUMNS * tile_side, i // SHEET_COLUMNS * tile_side))
    sheet.save(run_dir / "detail_boxes.jpg", quality=90)
    species = pd.read_csv(DATA / "classes.csv").set_index("class_id")["species"]
    listing = sample[["path", "species"]].assign(
        top_species=species.reindex(sample_top).to_numpy(),
        **{side: sample_boxes[:, i].round(3) for i, side in enumerate(("x1", "y1", "x2", "y2"))})
    listing.to_csv(run_dir / "detail_boxes.csv", index=False)
    return summary, listing


def run_pass(name, epochs, limit):
    """Every step above for one pass, in order; a failing step stops the pass."""
    started = time.time()
    cfg = recipe(RUN_ID if name == "full" else f"{RUN_ID}_{name}", epochs)
    run_dir = WORK / cfg["run_id"]
    final = train(cfg, run_dir, DATA, MASKS, SUBSET, preds_root=PREDS, limit=limit)
    quality, decision = verdicts(run_dir)
    r1 = scenery_test(run_dir, quality, limit)
    boxes, listing = box_report(run_dir, cfg, limit)
    row = {**robustness.arm_row(APPROACH, final, decision, r1), "bank_sha256": bank_sha256}
    row["gap"] = row["top1"] - row["top1_swapped"]
    write_json(run_dir / "arm_row.json", row)
    return SimpleNamespace(name=name, cfg=cfg, run_dir=run_dir, final=final, decision=decision,
                           r1=r1, boxes=boxes, listing=listing, row=row,
                           minutes=(time.time() - started) / 60)

## 7. The smoke pass

Every step on the first 2,000 photos of each set, for one epoch. Its scores mean nothing; it checks that every step runs, so a mistake stops the session before the long pass. The sets are sorted by species, and the verdicts need WHO-listed photos among the calibration ones, which the first 2,000 have.

In [10]:
assert {"inputs", "bank"} <= PASSED, f"checks that did not finish: {sorted({'inputs', 'bank'} - PASSED)}"
smoke = run_pass("smoke", 1, SMOKE_PHOTOS)
SMOKE_PASSED = True
pd.Series({"minutes": smoke.minutes, "macro-F1": smoke.row["macro_f1"],
           "WHO recall": smoke.row["who_recall_dr0"], "top-1 swapped": smoke.row["top1_swapped"],
           "scenery-test photos": smoke.r1["photos"]["val"],
           "detail share": pd.read_json(smoke.run_dir / "metrics.jsonl", lines=True)["detail_share"].iloc[-1],
           "boxes": smoke.boxes["photos"]}).to_frame("smoke").round(3)

M4_H2_smoke: read 7959 files (863 MB, 0 missing) into the page cache in 43 s
M4_H2_smoke on Tesla T4: 2000 training photos after caps, 2000 val, 2000 calibration, 451 unknown; subset 40196d19a68e; front-end predictions cover {'train': 1.0, 'val': 0.999, 'calibration': 0.9995, 'unknown': 1.0}


M4_H2_smoke epoch 1/1: macro-F1 0.190, top-1 0.072, top-3 0.114, loss 19.223, lr 9.53e-04, 45 photos/s, data wait 5%, epoch 44 s + eval 22 s


,smoke
minutes,4.295
macro-F1,0.190
WHO recall,0.955
top-1 swapped,0.058
scenery-test photos,2000.000
detail share,0.516
boxes,2000.000


## 8. The full pass

The same functions on the whole subset for the recipe's 20 epochs.

In [11]:
assert SMOKE_PASSED, "the smoke pass did not finish, so the full pass does not start"
if FULL:
    full = run_pass("full", None, None)
    FULL_PASSED = True
result = full if FULL else smoke
print(f"{result.cfg['run_id']}: {result.minutes:.0f} min, best epoch {result.final['run']['best_epoch']}")

M4_H2: read 49487 files (5404 MB, 0 missing) into the page cache in 68 s
M4_H2 on Tesla T4: 13611 training photos after caps, 17151 val, 7999 calibration, 451 unknown; subset 40196d19a68e; front-end predictions cover {'train': 0.9997795900374697, 'val': 0.9995335548947584, 'calibration': 0.9998749843730467, 'unknown': 1.0}
M4_H2 epoch 2/20: macro-F1 0.207, top-1 0.466, top-3 0.642, loss 14.657, lr 9.47e-04, 83 photos/s, data wait 1%, epoch 164 s + eval 117 s
M4_H2 epoch 4/20: macro-F1 0.283, top-1 0.545, top-3 0.710, loss 12.397, lr 8.96e-04, 84 photos/s, data wait 1%, epoch 161 s + eval 114 s
M4_H2 epoch 6/20: macro-F1 0.317, top-1 0.578, top-3 0.732, loss 10.904, lr 7.99e-04, 85 photos/s, data wait 1%, epoch 160 s + eval 114 s
M4_H2 epoch 8/20: macro-F1 0.325, top-1 0.590, top-3 0.740, loss 9.906, lr 6.68e-04, 84 photos/s, data wait 1%, epoch 161 s + eval 114 s
M4_H2 epoch 10/20: macro-F1 0.343, top-1 0.598, top-3 0.747, loss 9.345, lr 5.16e-04, 83 photos/s, data wait 1%, epoch 163 s

Validation scores during training, every second epoch:

In [12]:
curve = pd.read_json(result.run_dir / "metrics.jsonl", lines=True)
curve[["epoch", "macro_f1", "top1", "top3", "loss_total", "detail_share", "seconds", "eval_seconds"]]

,epoch,macro_f1,top1,top3,loss_total,detail_share,seconds,eval_seconds
0,2,0.207,0.466,0.642,14.657,0.557,164.238,117.247
1,4,0.283,0.545,0.710,12.397,0.486,161.062,113.947
2,6,0.317,0.578,0.732,10.904,0.462,159.954,113.761
3,8,0.325,0.590,0.740,9.906,0.491,161.200,113.874
4,10,0.343,0.598,0.747,9.345,0.528,163.022,113.976
5,12,0.355,0.603,0.753,8.724,0.580,165.477,113.818
6,14,0.372,0.625,0.767,8.483,0.538,163.237,113.752
7,16,0.383,0.627,0.768,8.109,0.505,161.862,113.638
8,18,0.381,0.624,0.766,8.030,0.495,161.571,113.672
9,20,0.377,0.628,0.768,8.086,0.467,160.367,114.263


The verdicts and the scenery test:

In [13]:
v, c = result.decision["verdict_metrics"], result.r1["conditions"]
pd.Series({"WHO recall": v["who_dangerous_recall"], "false alarms": v["false_alarm"],
           "we don't know": v["abstain_rate"],
           "as-is agreement with the saved run": result.r1["instrument_check"]["top1_agreement"],
           "top-1 as is": c["as_is"]["top1"],
           "top-1 as is, photos with a mask": c["swapped"]["as_is_same_photos"]["top1"],
           "top-1 snake blanked": c["blanked"]["top1"], "top-1 scenery swapped": c["swapped"]["top1"],
           "follows the donor's species": c["swapped"]["donor"]["follows_donor"]}).to_frame(result.cfg["run_id"])

,M4_H2
WHO recall,0.949
false alarms,0.299
we don't know,0.095
as-is agreement with the saved run,1.000
top-1 as is,0.627
"top-1 as is, photos with a mask",0.634
top-1 snake blanked,0.116
top-1 scenery swapped,0.541
follows the donor's species,0.013


## 9. Where the model looks

How much the detail boxes zoom on the validation photos: the box's share of the photo, how often it keeps (almost) the whole photo, and how often it reaches into a corner. Below, the 20 photos drawn into `detail_boxes.jpg`, with the model's top species.

In [14]:
display(pd.Series(result.boxes).rename("detail boxes on validation photos"))
result.listing.drop(columns="path")

photos               17151.000
median_area              0.450
whole_photo_share        0.164
corner_share             0.460
smallest_area            0.062
Name: detail boxes on validation photos, dtype: float64

,species,top_species,x1,y1,x2,y2
0,Psammophis condanarus,Psammophis schokari,0.0,0.0,1.0,1.0
1,Hebius khasiensis,Hebius khasiensis,0.2,0.2,0.9,0.7
2,Coelognathus flavolineatus,Naja naja,0.0,0.2,0.4,0.8
3,Boiga cyanea,Sahyadriophis beddomei,0.0,0.0,1.0,0.4
4,Laticauda colubrina,Laticauda colubrina,0.0,0.0,1.0,0.9
5,Sibynophis collaris,Sibynophis collaris,0.1,0.4,1.0,0.7
6,Gongylosoma calamaria,Ptyas korros,0.0,0.0,1.0,0.5
7,Boiga dightoni,Boiga dightoni,0.0,0.2,1.0,0.9
8,Argyrophis diardii,Indotyphlops braminus,0.5,0.4,1.0,0.7
9,Ahaetulla farnsworthi,Ahaetulla borealis,0.0,0.2,1.0,1.0


## 10. Time per frame

The detail crop runs the encoder twice, so it should cost about twice the time of one look. We time one 320 × 320 frame on this machine's processor, with and without the second look (median of 30 runs after 5 warm-up runs). The laptop's time per frame is measured with the same code on the laptop.

In [15]:
def ms_per_frame(net, side, runs=30, warmup=5):
    frame = torch.randn(1, 3, side, side)
    times = []
    with torch.inference_mode():
        for i in range(warmup + runs):
            started = time.perf_counter()
            net(frame)
            if i >= warmup:
                times.append((time.perf_counter() - started) * 1000)
    return statistics.median(times)


_, cpu_model, _ = load_checkpoint(result.run_dir, "cpu")
timing = {}
for label, on in (("one look", False), ("with the detail crop", True)):
    cpu_model.detail_crop = on
    timing[label] = ms_per_frame(cpu_model, result.cfg["input_size"])
timing_cpu = platform.processor() or platform.machine()
pd.Series(timing).rename(f"ms per frame on {timing_cpu}, {torch.get_num_threads()} threads").round(1)

one look                 89.5
with the detail crop    182.5
Name: ms per frame on x86_64, 2 threads, dtype: float64

## 11. Against the background winner

The run's row next to A1's, on the same photos and the same scenery bank. **Gap** is top-1 as is minus top-1 with the scenery swapped, on the photos with a usable mask: the smaller, the less the model leans on the scenery. The run's numbers are saved to `results/m4/heads/` under the run's name.

In [16]:
RESULTS = REPO / "results" / "m4" / "heads"
RESULTS.mkdir(parents=True, exist_ok=True)
summary = write_json(RESULTS / f"{result.cfg['run_id']}.json", {
    **result.row, "approach": APPROACH, "winner": WINNER,
    "settings": {name: result.cfg[name] for name in ("detail_crop", "detail_threshold", "detail_train_p")},
    "top3": result.final["head"]["top3"],
    "ms_per_frame": {"machine": timing_cpu, "threads": torch.get_num_threads(), **timing},
    "detail_boxes": result.boxes,
    "run": {name: result.final["run"][name] for name in ("best_epoch", "epochs", "train_photos", "val_photos")},
    "minutes": result.minutes})
names = {"macro_f1": "macro-F1", "who_recall_dr0": "WHO recall", "top1": "top-1, photos with a mask",
         "top1_swapped": "top-1 scenery swapped", "top1_blanked": "top-1 snake blanked", "gap": "gap",
         "swap_photos": "swapped photos"}
table = pd.DataFrame({f"{WINNER} (A1)": {names[k]: winner_row[k] for k in names},
                      result.cfg["run_id"]: {names[k]: summary[k] for k in names}})
table["difference"] = table.iloc[:, 1] - table.iloc[:, 0]
table

,M4_A1 (A1),M4_H2,difference
macro-F1,0.372,0.383,1.079e-02
WHO recall,0.945,0.949,3.754e-03
"top-1, photos with a mask",0.631,0.634,3.129e-03
top-1 scenery swapped,0.538,0.541,3.255e-03
top-1 snake blanked,0.140,0.116,-2.460e-02
gap,0.093,0.093,-1.252e-04
swapped photos,15977.000,15977.000,0.000e+00


*Note added after the run:* the cell above wrote the training summary (best epoch, epochs, photos) into the `"run"` field, overwriting `arm_row`'s run id. The committed `results/m4/heads/M4_H2.json` was rebuilt from the run's own `arm_row.json` and the file this cell wrote: `"run"` holds the run id again and the training summary sits under `"training"`, with no value changed.